In [ ]:
"""
Dummy EMG + MPU6050 Data Generator
===================================
Mimics the structure of NinaPro DB5's 5 subsets:
  1. PR    - Pattern Recognition (gesture classification)
  2. MVC   - Max Voluntary Contraction (normalization reference)
  3. 1-DoF - Single finger, varying force (regression)
  4. N-DoF - Multi-finger combinations (pre-training)
  5. Random - Unstructured movements (robustness)

Signal layout per sample:
  - 8 EMG channels  (forearm muscle electrodes)
  - 3 ACC channels  (accelerometer x, y, z)
  - 3 GYRO channels (gyroscope x, y, z)
  = 14 channels total, sampled at 200 Hz

Realism improvements over v1:
  - Irregular motor-unit burst envelope (not a clean sine hill)
  - Baseline wander: slow 0.1–0.5 Hz drift on every EMG channel
  - Inter-channel crosstalk: active channels bleed ~10–20 % onto neighbours
  - Amplitude dropout: occasional signal dip mid-gesture (~5 % probability)
  - IMU: Gaussian jitter spikes, sensor drift ramp, per-axis DC offset
  - 1-DoF force: asymmetric ramp + fatigue sag on the way down
"""

import numpy as np
import os

# ─────────────────────────────────────────────
# GLOBAL CONFIG
# ─────────────────────────────────────────────
SAMPLE_RATE     = 200       # Hz — samples per second
EMG_CHANNELS    = 8         # number of EMG electrodes
ACC_CHANNELS    = 3         # accelerometer axes (x, y, z)
GYRO_CHANNELS   = 3         # gyroscope axes (x, y, z)
TOTAL_CHANNELS  = EMG_CHANNELS + ACC_CHANNELS + GYRO_CHANNELS  # = 14

N_GESTURES      = 34        # PR subset: 34 hand gestures
N_FINGERS       = 5         # MVC/DoF subsets: 5 fingers
REPS_PER_CLASS  = 50        # how many repetitions per gesture/class

GESTURE_DURATION_S  = 1.0   # seconds — how long one gesture lasts
GESTURE_SAMPLES     = int(SAMPLE_RATE * GESTURE_DURATION_S)  # = 200 samples

OUTPUT_DIR = "dummy_ninapro_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

np.random.seed(42)          # reproducibility


# ─────────────────────────────────────────────
# NEW HELPER: baseline wander
# ─────────────────────────────────────────────
def baseline_wander(n_samples, amplitude=0.04):
    """
    Generate a slow sinusoidal drift (0.1–0.5 Hz) that rides underneath
    every EMG channel.  In real recordings this comes from skin-electrode
    impedance changes and respiration artefacts.

    Parameters
    ----------
    n_samples : int   — window length
    amplitude : float — how large the drift can be (default ±0.04)

    Returns
    -------
    wander : ndarray of shape (n_samples,)
    """
    # Pick a random low frequency (0.1 – 0.5 Hz) for this window
    freq = np.random.uniform(0.1, 0.5)
    # Random phase so not every channel drifts in sync
    phase = np.random.uniform(0, 2 * np.pi)
    t = np.linspace(0, GESTURE_DURATION_S, n_samples)
    # Scale by a random draw so amplitude varies rep-to-rep
    return amplitude * np.random.uniform(0.5, 1.5) * np.sin(2 * np.pi * freq * t + phase)


# ─────────────────────────────────────────────
# NEW HELPER: irregular burst envelope
# ─────────────────────────────────────────────
def irregular_envelope(n_samples):
    """
    Build a muscle-activation envelope that is NOT a perfect sine hill.

    Strategy
    --------
    1. Start with a smooth sine hill (the "ideal" envelope).
    2. Multiply by a slowly-varying random walk so the peak is uneven.
    3. Add small rapid fluctuations (~10–15 Hz) to mimic motor-unit
       recruitment jitter — real muscles don't fire perfectly smoothly.

    Returns
    -------
    envelope : ndarray of shape (n_samples,), values in [0, 1] range
    """
    # --- base hill (same as before) ---
    base = np.sin(np.linspace(0, np.pi, n_samples))  # smooth 0→1→0

    # --- slow modulation (3–6 Hz amplitude ripple) ---
    mod_freq  = np.random.uniform(3, 6)
    mod_phase = np.random.uniform(0, 2 * np.pi)
    t = np.linspace(0, GESTURE_DURATION_S, n_samples)
    # keep modulation subtle: ±15 % variation around 1.0
    slow_mod = 1.0 + 0.15 * np.sin(2 * np.pi * mod_freq * t + mod_phase)

    # --- fast jitter (motor-unit recruitment noise, ~10–15 Hz) ---
    jitter_freq = np.random.uniform(10, 15)
    jitter = 0.06 * np.sin(2 * np.pi * jitter_freq * t + np.random.uniform(0, 2 * np.pi))

    envelope = base * slow_mod + jitter

    # Clip so we don't go negative or shoot above 1.2
    return np.clip(envelope, 0, 1.2)


# ─────────────────────────────────────────────
# HELPER: generate one EMG burst  (v2 — realistic)
# ─────────────────────────────────────────────
def emg_burst(n_samples, active_channels, base_amplitude=0.5, noise_level=0.05):
    """
    Simulate realistic EMG signal for one gesture window.

    Additions vs v1
    ---------------
    • Irregular envelope (not a clean sine hill)
    • Per-channel baseline wander
    • Inter-channel crosstalk from active → nearby channels
    • Occasional mid-burst amplitude dropout (~5 % probability)

    Parameters
    ----------
    n_samples       : int   — number of time steps
    active_channels : list  — which of the 8 EMG channels are "firing"
    base_amplitude  : float — peak signal strength (0 to 1 scale)
    noise_level     : float — background noise level

    Returns
    -------
    emg : ndarray of shape (n_samples, EMG_CHANNELS)
    """
    t = np.linspace(0, GESTURE_DURATION_S, n_samples)

    # Start with Gaussian noise on all channels (baseline muscle activity)
    emg = np.random.normal(0, noise_level, (n_samples, EMG_CHANNELS))

    # Add per-channel baseline wander to ALL channels
    # (even inactive ones drift in real recordings)
    for ch in range(EMG_CHANNELS):
        emg[:, ch] += baseline_wander(n_samples, amplitude=noise_level * 0.8)

    for ch in active_channels:
        # --- irregular envelope instead of a clean sine ---
        envelope = irregular_envelope(n_samples)

        # --- amplitude with per-rep variability ---
        amplitude = base_amplitude * np.random.uniform(0.75, 1.25)

        # --- high-frequency EMG oscillation (motor unit firings) ---
        frequency = np.random.uniform(80, 150)
        oscillation = np.sin(2 * np.pi * frequency * t)

        # --- amplitude dropout: ~5 % chance of a brief signal dip ---
        dropout_mask = np.ones(n_samples)
        if np.random.rand() < 0.05:
            # Drop out a random 10–25 % chunk of the window
            drop_start = np.random.randint(0, int(n_samples * 0.75))
            drop_len   = np.random.randint(n_samples // 10, n_samples // 4)
            drop_end   = min(drop_start + drop_len, n_samples)
            # Smooth the edges of the dropout with a cosine taper
            dropout_mask[drop_start:drop_end] = 0.1
            # Smooth transition in/out (avoid hard step edges)
            taper = 10
            for i in range(taper):
                if drop_start + i < n_samples:
                    dropout_mask[drop_start + i] = 0.1 + 0.9 * (i / taper)
                if drop_end - taper + i < n_samples:
                    dropout_mask[drop_end - taper + i] = 1.0 - 0.9 * (i / taper)

        # Combine everything onto this active channel
        emg[:, ch] += amplitude * envelope * oscillation * dropout_mask

        # --- inter-channel crosstalk ---
        # Active channel bleeds onto its immediate neighbours (vol-conduction through tissue)
        # Bleed = 10–20 % of the active signal, decaying with distance
        bleed_ratio = np.random.uniform(0.10, 0.20)
        for neighbour in [ch - 1, ch + 1]:
            if 0 <= neighbour < EMG_CHANNELS:
                # Neighbour gets a phase-shifted, attenuated copy
                phase_shift = np.random.uniform(0.01, 0.05)   # small time lag in seconds
                shift_samples = int(phase_shift * SAMPLE_RATE)
                bleed_signal = amplitude * envelope * oscillation * dropout_mask * bleed_ratio
                # Shift the bleed signal slightly in time
                if shift_samples > 0:
                    bleed_signal = np.roll(bleed_signal, shift_samples)
                    bleed_signal[:shift_samples] = 0
                emg[:, neighbour] += bleed_signal

    return emg


# ─────────────────────────────────────────────
# HELPER: generate MPU6050 (ACC + GYRO)  (v2 — realistic)
# ─────────────────────────────────────────────
def imu_signal(n_samples, gesture_id, noise_level=0.02):
    """
    Simulate realistic IMU signal for a gesture.

    Additions vs v1
    ---------------
    • Random DC offset per axis (electrode placement bias)
    • Linear sensor drift ramp (temperature / time drift)
    • Occasional 1–3 sample jitter spikes (vibration / cable tap)
    • Gyro: asymmetric settle (doesn't return cleanly to zero)

    Returns
    -------
    imu : ndarray of shape (n_samples, 6)
          columns: [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
    """
    orientation_map = {
        0: [0.1,  0.0,  1.0],
        1: [0.5,  0.2,  0.8],
        2: [0.8,  0.1,  0.5],
        3: [0.0,  0.7,  0.7],
        4: [-0.3, 0.4,  0.9],
        5: [0.2, -0.5,  0.8],
    }
    acc_mean = orientation_map[gesture_id % 6]

    t = np.linspace(0, 2 * np.pi, n_samples)

    # --- accelerometer ---
    acc = np.zeros((n_samples, ACC_CHANNELS))
    for i in range(ACC_CHANNELS):
        # Random DC bias (mounting offset, ~±0.05 g)
        dc_offset = np.random.uniform(-0.05, 0.05)

        # Linear drift across the window (slow thermal drift, ~±0.03 g max)
        drift = np.linspace(0, np.random.uniform(-0.03, 0.03), n_samples)

        acc[:, i] = (
            acc_mean[i]
            + dc_offset
            + drift
            + 0.05 * np.sin(t + i)                              # wobble (same as v1)
            + np.random.normal(0, noise_level, n_samples)        # Gaussian noise
        )

    # --- inject 1–3 jitter spikes on random accelerometer axes ---
    n_spikes = np.random.randint(1, 4)
    for _ in range(n_spikes):
        spike_axis   = np.random.randint(0, ACC_CHANNELS)
        spike_sample = np.random.randint(0, n_samples)
        spike_amp    = np.random.uniform(0.1, 0.3) * np.random.choice([-1, 1])
        # Spike lasts 1–2 samples
        spike_len = np.random.randint(1, 3)
        acc[spike_sample:spike_sample + spike_len, spike_axis] += spike_amp

    # --- gyroscope ---
    gyro = np.zeros((n_samples, GYRO_CHANNELS))
    for i in range(GYRO_CHANNELS):
        spike_len = n_samples // 5
        spike = np.zeros(n_samples)
        peak_val = np.random.uniform(-30, 30)
        spike[:spike_len] = np.linspace(0, peak_val, spike_len)

        # Asymmetric decay: real gyro doesn't settle cleanly to zero
        # It overshoots slightly and then damps — model this as:
        #   a fast decay that leaves a small residual offset
        residual = np.random.uniform(-2, 2)   # leftover angular velocity (deg/s)
        decay_len = n_samples - spike_len
        spike[spike_len:] = np.linspace(peak_val, residual, decay_len)

        # Add low-level vibration noise on gyro (much noisier than acc in practice)
        gyro[:, i] = spike + np.random.normal(0, noise_level * 8, n_samples)

    return np.hstack([acc, gyro])   # shape: (n_samples, 6)


# ─────────────────────────────────────────────
# SUBSET 1: Pattern Recognition (PR)
# ─────────────────────────────────────────────
def generate_pr_subset():
    """
    34 gestures × REPS_PER_CLASS repetitions.
    Each sample: GESTURE_SAMPLES × TOTAL_CHANNELS signal window + label.
    """
    print("Generating Subset 1: Pattern Recognition (PR)...")

    gesture_channel_map = {
        g: list(np.random.choice(EMG_CHANNELS, size=np.random.randint(2, 5), replace=False))
        for g in range(N_GESTURES)
    }

    all_signals = []
    all_labels  = []

    for gesture_id in range(N_GESTURES):
        active_chs = gesture_channel_map[gesture_id]

        for rep in range(REPS_PER_CLASS):
            emg = emg_burst(
                n_samples=GESTURE_SAMPLES,
                active_channels=active_chs,
                base_amplitude=0.5 + gesture_id * 0.01,
            )
            imu    = imu_signal(GESTURE_SAMPLES, gesture_id)
            window = np.hstack([emg, imu])

            all_signals.append(window)
            all_labels.append(gesture_id)

    X = np.array(all_signals)
    y = np.array(all_labels)

    np.save(f"{OUTPUT_DIR}/pr_signals.npy", X)
    np.save(f"{OUTPUT_DIR}/pr_labels.npy",  y)

    print(f"  ✓ Signals shape : {X.shape}  → (samples, time_steps, channels)")
    print(f"  ✓ Labels shape  : {y.shape}  → (samples,)")
    print(f"  ✓ Saved to      : {OUTPUT_DIR}/pr_signals.npy\n")

    return X, y


# ─────────────────────────────────────────────
# SUBSET 2: MVC (Max Voluntary Contraction)
# ─────────────────────────────────────────────
def generate_mvc_subset():
    """
    5 fingers × REPS_PER_CLASS repetitions.
    Each sample is a MAX effort contraction — strongest possible signal.
    Used ONLY for normalization, not classification training.
    """
    print("Generating Subset 2: MVC (Max Voluntary Contraction)...")

    finger_channel_map = {
        0: [0, 1], 1: [1, 2], 2: [2, 3], 3: [3, 4], 4: [4, 5],
    }

    mvc_values  = []
    all_signals = []
    all_labels  = []

    for finger_id in range(N_FINGERS):
        active_chs = finger_channel_map[finger_id]

        for rep in range(REPS_PER_CLASS):
            # MVC = max effort, so amplitude = 1.0, but real maxes still jitter
            emg = emg_burst(
                n_samples=GESTURE_SAMPLES,
                active_channels=active_chs,
                base_amplitude=1.0,
                noise_level=0.03,    # slightly more noise than v1 — hands shake at max effort
            )
            imu    = imu_signal(GESTURE_SAMPLES, finger_id)
            window = np.hstack([emg, imu])

            mvc_values.append(np.max(np.abs(emg), axis=0))
            all_signals.append(window)
            all_labels.append(finger_id)

    X   = np.array(all_signals)
    y   = np.array(all_labels)
    mvc = np.array(mvc_values)

    np.save(f"{OUTPUT_DIR}/mvc_signals.npy", X)
    np.save(f"{OUTPUT_DIR}/mvc_labels.npy",  y)
    np.save(f"{OUTPUT_DIR}/mvc_values.npy",  mvc)

    print(f"  ✓ Signals shape : {X.shape}")
    print(f"  ✓ MVC values    : {mvc.shape} → (reps, emg_channels)")
    print(f"  ✓ Saved to      : {OUTPUT_DIR}/mvc_signals.npy\n")

    return X, y, mvc


# ─────────────────────────────────────────────
# SUBSET 3: 1-DoF (Single Finger, Varying Force)
# ─────────────────────────────────────────────
def generate_1dof_subset():
    """
    5 fingers × REPS_PER_CLASS repetitions.
    Force varies CONTINUOUSLY — regression problem.

    Additions vs v1
    ---------------
    • Asymmetric ramp: rise time ≠ fall time (real presses aren't symmetric)
    • Fatigue sag: on the descent, force doesn't return cleanly to 0
    • Tremor is slightly louder to match real hand tremor range
    """
    print("Generating Subset 3: 1-DoF (Single Finger, Varying Force)...")

    finger_channel_map = {
        0: [0, 1], 1: [1, 2], 2: [2, 3], 3: [3, 4], 4: [4, 5],
    }

    all_signals = []
    all_forces  = []
    all_labels  = []

    for finger_id in range(N_FINGERS):
        active_chs = finger_channel_map[finger_id]

        for rep in range(REPS_PER_CLASS):
            # --- asymmetric ramp ---
            # Rise time: 30–60 % of the window (faster to press than release)
            rise_frac = np.random.uniform(0.30, 0.60)
            rise_len  = int(GESTURE_SAMPLES * rise_frac)
            fall_len  = GESTURE_SAMPLES - rise_len

            rise = np.linspace(0.0, 1.0, rise_len)

            # Fatigue sag: don't return all the way to 0; leave a small residual
            fatigue_floor = np.random.uniform(0.0, 0.08)
            fall = np.linspace(1.0, fatigue_floor, fall_len)

            force_profile = np.concatenate([rise, fall])

            # --- tremor (~8–12 Hz natural hand tremor, slightly louder) ---
            t = np.linspace(0, GESTURE_DURATION_S, GESTURE_SAMPLES)
            tremor_freq = np.random.uniform(8, 12)
            tremor = 0.05 * np.sin(2 * np.pi * tremor_freq * t + np.random.uniform(0, 2 * np.pi))
            force_profile = np.clip(force_profile + tremor, 0, 1)

            # EMG amplitude follows force
            emg = np.zeros((GESTURE_SAMPLES, EMG_CHANNELS))
            for i in range(GESTURE_SAMPLES):
                emg[i, :] = np.random.normal(0, 0.02, EMG_CHANNELS)
                for ch in active_chs:
                    emg[i, ch] += force_profile[i] * np.random.uniform(0.8, 1.0)

            # Add baseline wander on top
            for ch in range(EMG_CHANNELS):
                emg[:, ch] += baseline_wander(GESTURE_SAMPLES, amplitude=0.015)

            imu    = imu_signal(GESTURE_SAMPLES, finger_id)
            window = np.hstack([emg, imu])

            all_signals.append(window)
            all_forces.append(force_profile)
            all_labels.append(finger_id)

    X = np.array(all_signals)
    F = np.array(all_forces)
    y = np.array(all_labels)

    np.save(f"{OUTPUT_DIR}/dof1_signals.npy", X)
    np.save(f"{OUTPUT_DIR}/dof1_forces.npy",  F)
    np.save(f"{OUTPUT_DIR}/dof1_labels.npy",  y)

    print(f"  ✓ Signals shape : {X.shape}")
    print(f"  ✓ Forces shape  : {F.shape}  → (samples, time_steps) — continuous force target")
    print(f"  ✓ Saved to      : {OUTPUT_DIR}/dof1_signals.npy\n")

    return X, F, y


# ─────────────────────────────────────────────
# SUBSET 4: N-DoF (Multi-Finger Combinations)
# ─────────────────────────────────────────────
def generate_ndof_subset():
    """
    All combinations of 2-3 fingers activating together.
    Used for CNN pre-training (transfer learning).
    """
    print("Generating Subset 4: N-DoF (Multi-Finger Combinations)...")

    combinations = []
    for i in range(N_FINGERS):
        for j in range(i+1, N_FINGERS):
            combinations.append((i, j))
            for k in range(j+1, N_FINGERS):
                combinations.append((i, j, k))

    finger_channel_map = {
        0: [0, 1], 1: [1, 2], 2: [2, 3], 3: [3, 4], 4: [4, 5],
    }

    all_signals = []
    all_labels  = []

    for combo_id, combo in enumerate(combinations):
        active_chs = list(set(
            ch for finger in combo for ch in finger_channel_map[finger]
        ))

        for rep in range(REPS_PER_CLASS):
            emg = emg_burst(
                n_samples=GESTURE_SAMPLES,
                active_channels=active_chs,
                base_amplitude=0.4 + 0.1 * len(combo),
            )
            imu    = imu_signal(GESTURE_SAMPLES, combo_id)
            window = np.hstack([emg, imu])

            all_signals.append(window)
            all_labels.append(combo_id)

    X = np.array(all_signals)
    y = np.array(all_labels)

    np.save(f"{OUTPUT_DIR}/ndof_signals.npy", X)
    np.save(f"{OUTPUT_DIR}/ndof_labels.npy",  y)
    np.save(f"{OUTPUT_DIR}/ndof_combo_map.npy", np.array(combinations, dtype=object))

    print(f"  ✓ Total combinations : {len(combinations)} (2-finger + 3-finger)")
    print(f"  ✓ Signals shape      : {X.shape}")
    print(f"  ✓ Labels shape       : {y.shape}")
    print(f"  ✓ Saved to           : {OUTPUT_DIR}/ndof_signals.npy\n")

    return X, y, combinations


# ─────────────────────────────────────────────
# SUBSET 5: Random Task (Unstructured Movement)
# ─────────────────────────────────────────────
def generate_random_subset():
    """
    Random, unstructured EMG + IMU signals.
    No gesture label — used for robustness training.
    """
    print("Generating Subset 5: Random Task (Unstructured Movements)...")

    N_RANDOM_SAMPLES = 500

    all_signals = []

    for i in range(N_RANDOM_SAMPLES):
        n_active = np.random.randint(0, EMG_CHANNELS + 1)

        if n_active == 0:
            # Idle: pure noise + wander
            emg = np.random.normal(0, 0.03, (GESTURE_SAMPLES, EMG_CHANNELS))
            for ch in range(EMG_CHANNELS):
                emg[:, ch] += baseline_wander(GESTURE_SAMPLES, amplitude=0.02)
        else:
            active_chs = np.random.choice(EMG_CHANNELS, size=n_active, replace=False).tolist()
            emg = emg_burst(
                n_samples=GESTURE_SAMPLES,
                active_channels=active_chs,
                base_amplitude=np.random.uniform(0.05, 0.25),
                noise_level=0.08,
            )

        # Random IMU — no pattern, but still physically plausible
        imu = np.random.normal(0, 0.12, (GESTURE_SAMPLES, ACC_CHANNELS + GYRO_CHANNELS))
        imu[:, 2] += np.random.uniform(0.7, 1.0)   # Z gravity component

        # Add a few jitter spikes to IMU as well
        for _ in range(np.random.randint(0, 3)):
            ax  = np.random.randint(0, ACC_CHANNELS)
            idx = np.random.randint(0, GESTURE_SAMPLES)
            imu[idx, ax] += np.random.uniform(-0.4, 0.4)

        window = np.hstack([emg, imu])
        all_signals.append(window)

    X = np.array(all_signals)
    y = np.full(N_RANDOM_SAMPLES, -1)

    np.save(f"{OUTPUT_DIR}/random_signals.npy", X)
    np.save(f"{OUTPUT_DIR}/random_labels.npy",  y)

    print(f"  ✓ Signals shape : {X.shape}")
    print(f"  ✓ Labels        : all -1 (no gesture / negative class)")
    print(f"  ✓ Saved to      : {OUTPUT_DIR}/random_signals.npy\n")

    return X, y


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("  NinaPro-Style Dummy Data Generator  (v2 — realistic)")
    print("  Channels: 8 EMG + 3 ACC + 3 GYRO = 14 total")
    print(f"  Sample rate: {SAMPLE_RATE} Hz | Window: {GESTURE_DURATION_S}s ({GESTURE_SAMPLES} samples)")
    print("=" * 55)
    print()

    pr_X,   pr_y             = generate_pr_subset()
    mvc_X,  mvc_y,  mvc_vals = generate_mvc_subset()
    dof1_X, dof1_F, dof1_y  = generate_1dof_subset()
    ndof_X, ndof_y, combos   = generate_ndof_subset()
    rand_X, rand_y           = generate_random_subset()

    print("=" * 55)
    print("  ALL SUBSETS GENERATED SUCCESSFULLY")
    print("=" * 55)
    print()
    print("  Files saved:")
    for f in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, f)
        size_kb = os.path.getsize(path) / 1024
        print(f"    {f:<35} {size_kb:>7.1f} KB")

    print()
    print("  How to load in your training script:")
    print("    import numpy as np")
    print(f"    X = np.load('{OUTPUT_DIR}/pr_signals.npy')")
    print(f"    y = np.load('{OUTPUT_DIR}/pr_labels.npy')")
    print(f"    # X.shape → (1700, 200, 14)")
    print(f"    # y.shape → (1700,)")

  NinaPro-Style Dummy Data Generator
  Channels: 8 EMG + 3 ACC + 3 GYRO = 14 total
  Sample rate: 200 Hz | Window: 1.0s (200 samples)

Generating Subset 1: Pattern Recognition (PR)...
  ✓ Signals shape : (1700, 200, 14)  → (samples, time_steps, channels)
  ✓ Labels shape  : (1700,)  → (samples,)
  ✓ Saved to      : dummy_ninapro_data/pr_signals.npy

Generating Subset 2: MVC (Max Voluntary Contraction)...
  ✓ Signals shape : (250, 200, 14)
  ✓ MVC values    : (250, 8) → (reps, emg_channels) — use mean per finger for normalization
  ✓ Saved to      : dummy_ninapro_data/mvc_signals.npy

Generating Subset 3: 1-DoF (Single Finger, Varying Force)...
  ✓ Signals shape : (250, 200, 14)
  ✓ Forces shape  : (250, 200)  → (samples, time_steps) — continuous force target
  ✓ Saved to      : dummy_ninapro_data/dof1_signals.npy

Generating Subset 4: N-DoF (Multi-Finger Combinations)...
  ✓ Total combinations : 20 (2-finger + 3-finger)
  ✓ Signals shape      : (1000, 200, 14)
  ✓ Labels shape       : 